In [9]:
import os
from typing import List, Dict, Any
import pandas as pd

In [7]:
from langchain_core.documents import Document
from langchain_text_splitters import (RecursiveCharacterTextSplitter,CharacterTextSplitter,TokenTextSplitter)
print("set up is completed !!")

d:\UdemyRAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Understand Document Structure

In [14]:
## create a simple document
doc = Document(
    page_content="This is the main text content that will be embedded and searched",
    metadata = {
        "source" : " example.txt",
        "page" : 1,
        "author" : "Narsimha",
        "date_created" : "2026-08-11",
        "custom_filed" : "any_value"
    }
)
print("Document Stracture")
print(f"Content : {doc.page_content}")
print(f"Metadata : {doc.metadata}")

print("\n Metadata is  crucial for")
print("- Filtering  search results")
print(" - Tracking document sources")
print("- Providing context in responses")
print("- Debugging and aduting")

Document Stracture
Content : This is the main text content that will be embedded and searched
Metadata : {'source': ' example.txt', 'page': 1, 'author': 'Narsimha', 'date_created': '2026-08-11', 'custom_filed': 'any_value'}

 Metadata is  crucial for
- Filtering  search results
 - Tracking document sources
- Providing context in responses
- Debugging and aduting


In [15]:
print(type(doc))

<class 'langchain_core.documents.base.Document'>


## Text Files(.txt) - The simplest case {#2-text-files}

In [16]:
import os
os.makedirs("data/text_files",exist_ok=True)

In [26]:
simple_texts = {
    "data/text_files/python_intro.txt" :"""
Python is a high-level, interpreted programming language known for its simple and readable syntax. It is widely used in web development, data analysis, artificial intelligence, machine learning, automation, scripting, and software development.

Python is beginner-friendly because its syntax is easy to understand and closely resembles natural language. It supports important programming concepts such as variables, data types, conditions, loops, functions, classes, and object-oriented programming.

Python also has a large collection of libraries and frameworks, making it possible to build applications efficiently. Popular Python technologies include Django, Flask, FastAPI, Pandas, NumPy, and TensorFlow.

    
    """,
    "data/text_files/langchain.txt":"""A
    LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number, or document type.

Document loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.

The main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.

    
    """

}

for filepath,content in simple_texts.items():
    with open(filepath,'w',encoding='utf-8') as f:
        f.write(content)
print("sample file is created")

sample file is created


## TextLoader - Read Single file

In [25]:
from langchain_community.document_loaders import TextLoader
# Load a single text file
loader = TextLoader("data/text_files/python_intro.txt",encoding='utf-8')
documents = loader.load()
print(type(documents))
print(documents)
print("********************************")
print(f"Loaded : {len(documents)}")
print(f"Content Preview {documents[0].page_content[:100]}")
print(f"MetaData : {documents[0].metadata}")


<class 'list'>
[Document(metadata={'source': 'data/text_files/python_intro.txt'}, page_content='\nPython is a high-level, interpreted programming language known for its simple and readable syntax. It is widely used in web development, data analysis, artificial intelligence, machine learning, automation, scripting, and software development.\n\nPython is beginner-friendly because its syntax is easy to understand and closely resembles natural language. It supports important programming concepts such as variables, data types, conditions, loops, functions, classes, and object-oriented programming.\n\nPython also has a large collection of libraries and frameworks, making it possible to build applications efficiently. Popular Python technologies include Django, Flask, FastAPI, Pandas, NumPy, and TensorFlow.\n\n\n    ')]
********************************
Loaded : 1
Content Preview 
Python is a high-level, interpreted programming language known for its simple and readable syntax. 
MetaData : {'s

## DirectoryLoader - Multiple Text file

In [35]:
from langchain_community.document_loaders import DirectoryLoader
dir_loads = DirectoryLoader(
    "data/text_files",
    glob = "**/*.txt", ## patterns to match files
    loader_cls = TextLoader, ## Loader class to use
    loader_kwargs = {'encoding':'utf-8'},
    show_progress = True
)
documents = dir_loads.load()
print(documents)
for i,doc in enumerate(documents):
    print(f"\n Document {i+1}")
    print(f"Sourec : {doc.metadata['source']}")
    print(f" Length : {len(doc.page_content)} characters")
    print(f"content : {doc.page_content[:100]}")


print("\n DirectorLoader Charcterstics")
print("Advantages")
print("- Load multiple files at once")
print("- supports Glob Parameters")
print("- Recursive directory scanning")

print("\n Disadvantages")
print("- All files must be in same type")
print("- Limited error handling per file")
print("- can be memory intensive for large files")



100%|██████████| 2/2 [00:00<00:00, 695.92it/s]

[Document(metadata={'source': 'data\\text_files\\langchain.txt'}, page_content='A\n    LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number, or document type.\n\nDocument loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.\n\nThe main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.\n\n\n    '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='\nPython is a high-level, interpreted programming language known for its simple and readable syntax. It is widely 

## Text Splitting Statergies

In [36]:
## Diffrent text splitting statergies
from langchain_text_splitters import (RecursiveCharacterTextSplitter,
 CharacterTextSplitter,
 TokenTextSplitter)
print(documents)


[Document(metadata={'source': 'data\\text_files\\langchain.txt'}, page_content='A\n    LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number, or document type.\n\nDocument loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.\n\nThe main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.\n\n\n    '), Document(metadata={'source': 'data\\text_files\\python_intro.txt'}, page_content='\nPython is a high-level, interpreted programming language known for its simple and readable syntax. It is widely 

In [37]:
### Method 1 - Character Text Splitter
text = documents[0].page_content
print(text)

A
    LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number, or document type.

Document loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.

The main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.


    


In [56]:
# Method 1: Chrater Based Splitting
print(" Chracter Text Splitter")
char_splitter = CharacterTextSplitter(
    separator="\n", # split on newlines use to see " " (space to see over lap)
    chunk_size=200, # Max chuck size in character
    chunk_overlap=20, # overlap between chunks
    length_function=len # How to measure chunks
)
char_chunks = char_splitter.split_text(text)
print(f"created {len(char_chunks)} count")
print(f" First chunk {char_chunks[0][:100]}")
print(char_chunks[0])
print("----------------")
print(char_chunks[1])
print("----------------")
print(char_chunks[2])
print("----------------")
print(char_chunks[3])


Created a chunk of size 274, which is longer than the specified 200
Created a chunk of size 280, which is longer than the specified 200


 Chracter Text Splitter
created 4 count
 First chunk A
A
----------------
LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number, or document type.
----------------
Document loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.
----------------
The main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.


In [ ]:
# Method2 - Recursive Character text splitter
print("Recursive Character text splitter")
recursive_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n","\n"," ",""], # split on newlines
    chunk_size=200, # Max chuck size in character
    chunk_overlap=20, # overlap between chunks
    length_function=len # How to measure chunks
)
recursive_chucks = recursive_splitter.split_text(text)
print(f"Created {len(recursive_chucks)} created")
print(recursive_chucks[0])
print("----------------")
print(recursive_chucks[1])
print("----------------")
print(recursive_chucks[2])
print("----------------")
print(recursive_chucks[3])
print("----------------")
print(recursive_chucks[4])
print("----------------")
print(recursive_chucks[5])
 


Recursive Character text splitter
Created 6 created
A
----------------
LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along
----------------
actual text along with metadata such as the file name, source, page number, or document type.
----------------
Document loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the data, the documents can be passed to
----------------
can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.
----------------
The main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.


In [ ]:
# to view the over lap
sample_text = "Python is a powerful and versatile programming language that is widely used in modern software development. It was designed with simplicity and readability in mind, making it a popular choice for both beginners and experienced developers. Python allows developers to build applications with fewer lines of code compared to many other programming languages."

recursive_splitter = RecursiveCharacterTextSplitter(
    separators=[" "], # split on newlines
    chunk_size=200, # Max chuck size in character
    chunk_overlap=20, # overlap between chunks
    length_function=len # How to measure chunks
)
re_splitter = recursive_splitter.split_text(sample_text)
print(f"\n simple text example - {len(re_splitter)} chunks:\n")
for i in range(len(re_splitter)-1):
    print(f"chunk {i+1}: '{re_splitter[i]}' ")
    print(f"chunk {i+2}: '{re_splitter[i+1]}' ")
 

    print()





 simple text example - 2 chunks:

chunk 1: 'Python is a powerful and versatile programming language that is widely used in modern software development. It was designed with simplicity and readability in mind, making it a popular choice for both' 
chunk 2: 'choice for both beginners and experienced developers. Python allows developers to build applications with fewer lines of code compared to many other programming languages.' 



In [60]:
# Method 3 - token text splitting
print('\n Token text splitter')
token_splitter = TokenTextSplitter(
    chunk_size=50, # size in tokens (not characters)
    chunk_overlap=10
)
token_chunks = token_splitter.split_text(text)
print(f" created {len(token_chunks)} chunks ")
print(token_chunks[0])
print("----------------")
print(token_chunks[1])
print("----------------")
print(token_chunks[2])
print("----------------")
print(token_chunks[3])



 Token text splitter
 created 4 chunks 
A
    LangChain Document Loaders are components used to load data from different sources into a standard document format that LangChain can process. A document generally contains the actual text along with metadata such as the file name, source, page number
----------------
 such as the file name, source, page number, or document type.

Document loaders can load information from sources such as PDF files, text files, Word documents, web pages, CSV files, and other data sources. After loading the
----------------
 files, and other data sources. After loading the data, the documents can be passed to text splitters, embedding models, and vector databases as part of a RAG pipeline.

The main purpose of a Document Loader is to convert
----------------
 main purpose of a Document Loader is to convert external data into LangChain Document objects so that the data can be processed consistently.


    


In [61]:
# Comparision
print("\n📊 Text Splitting Methods Comparison:")
 
print("\nCharacterTextSplitter:")
print("    ✅ Simple and predictable")
print("    ✅ Good for structured text")
print("    ❌ May break mid-sentence")
print("    Use when: Text has clear delimiters")
 
print("\nRecursiveCharacterTextSplitter:")
print("    ✅ Respects text structure")
print("    ✅ Tries multiple separators")
print("    ✅ Best general-purpose splitter")
print("    ❌ Slightly more complex")
print("    Use when: Default choice for most texts")
 
print("\nTokenTextSplitter:")
print("    ✅ Respects model token limits")
print("    ✅ More accurate for embeddings")
print("    ❌ Slower than character-based")
print("    Use when: Working with token-limited models")


📊 Text Splitting Methods Comparison:

CharacterTextSplitter:
    ✅ Simple and predictable
    ✅ Good for structured text
    ❌ May break mid-sentence
    Use when: Text has clear delimiters

RecursiveCharacterTextSplitter:
    ✅ Respects text structure
    ✅ Tries multiple separators
    ✅ Best general-purpose splitter
    ❌ Slightly more complex
    Use when: Default choice for most texts

TokenTextSplitter:
    ✅ Respects model token limits
    ✅ More accurate for embeddings
    ❌ Slower than character-based
    Use when: Working with token-limited models
